### Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

#### Convert to document data structure

In [4]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: 3.L4_L10 Module 2_Intelligent Agents Final.pdf
  ✓ Loaded 59 pages

Processing: L1_L3 Introduction & History.pdf
  ✓ Loaded 51 pages

Processing: L0-Course introduction.pdf
  ✓ Loaded 22 pages

Total documents loaded: 132


#### Chunking

In [7]:
def split_documents(documents,chunk_size=500,chunk_overlap=50):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [8]:
chunks = split_documents(all_pdf_documents)
chunks

Split 132 documents into 142 chunks

Example chunk:
Content: INTELLIGENT AGENTS...
Metadata: {'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-01-28T09:53:57+05:30', 'title': 'INTELLIGENT AGENTS', 'author': 'MAHE', 'moddate': '2026-01-28T09:53:57+05:30', 'source': '../data/pdf/3.L4_L10 Module 2_Intelligent Agents Final.pdf', 'total_pages': 59, 'page': 0, 'page_label': '1', 'source_file': '3.L4_L10 Module 2_Intelligent Agents Final.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-01-28T09:53:57+05:30', 'title': 'INTELLIGENT AGENTS', 'author': 'MAHE', 'moddate': '2026-01-28T09:53:57+05:30', 'source': '../data/pdf/3.L4_L10 Module 2_Intelligent Agents Final.pdf', 'total_pages': 59, 'page': 0, 'page_label': '1', 'source_file': '3.L4_L10 Module 2_Intelligent Agents Final.pdf', 'file_type': 'pdf'}, page_content='INTELLIGENT AGENTS'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-01-28T09:53:57+05:30', 'title': 'INTELLIGENT AGENTS', 'author': 'MAHE', 'moddate': '2026-01-28T09:53:57+05:30', 'source': '../data/pdf/3.L4_L10 Module 2_Intelligent Agents Final.pdf', 'total_pages': 59, 'page': 1, 'page_label': '2', 'source_file': '3.L4_L10 Module 2_Intelligent Agents Final.pdf', 'file_type': 'pdf'}, page_content='AGENT 

#### Embedding and vectorDB

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/krishrohilla/Desktop/MindzKonnected/Tutorial/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1293.05it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/lb/1wnz8jsx11gbw_8n15b7gcbc0000gn/T/ipykernel_1281/540119965.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [12]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
           
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
       
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
         
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
           
            documents_text.append(doc.page_content)
            
         
            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [14]:
# Convert text to embeddings
texts=[doc.page_content for doc in chunks]

# Generate embeddings

embeddings = embedding_manager.generate_embeddings(texts)

# store in vectorDB
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 142 texts...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches: 100%|██████████| 5/5 [00:21<00:00,  4.26s/it]

Generated embeddings with shape: (142, 384)
Adding 142 documents to vector store...
Successfully added 142 documents to vector store
Total documents in collection: 142


### Retriever Pipeline From VectorStore

In [15]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [16]:
rag_retriever.retrieve("What is percept sequence")

Retrieving documents for query: 'What is percept sequence'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:11<00:00, 11.76s/it]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_adaffc7a_3',
  'content': 'SIMPLE TERMS  -- [PAGE]\n• PERCEPT\n• AGENT’S PERCEPTUAL INPUTS AT ANY GIVEN INSTANT\n• PERCEPT SEQUENCE\n• COMPLETE HISTORY OF EVERYTHING THAT THE AGENT HAS \nEVER PERCEIVED.\n• ACTION\n• AN OPERATION INVOLVING AN ACTUATOR\n• ACTIONS CAN BE GROUPED INTO ACTION SEQUENCES',
  'metadata': {'title': 'INTELLIGENT AGENTS',
   'source_file': '3.L4_L10 Module 2_Intelligent Agents Final.pdf',
   'creator': 'Microsoft® PowerPoint® for Microsoft 365',
   'moddate': '2026-01-28T09:53:57+05:30',
   'total_pages': 59,
   'file_type': 'pdf',
   'author': 'MAHE',
   'creationdate': '2026-01-28T09:53:57+05:30',
   'page_label': '4',
   'producer': 'Microsoft® PowerPoint® for Microsoft 365',
   'doc_index': 3,
   'content_length': 263,
   'page': 3,
   'source': '../data/pdf/3.L4_L10 Module 2_Intelligent Agents Final.pdf'},
  'similarity_score': 0.12502604722976685,
  'distance': 0.8749739527702332,
  'rank': 1},
 {'id': 'doc_dde4be14_21',
  'content': 'LEARNING\

In [17]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

### Integration Vectordb Context pipeline With LLM output

In [32]:

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="qwen/qwen3-32b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [33]:
answer=rag_simple("What is percept sequence?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is percept sequence?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.81it/s]


Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
<think>
Okay, let's see. The user is asking for the definition of "percept sequence." I need to look at the provided context to find the answer.

Looking at the context under the "PERCEPT SEQUENCE" section, it says: "COMPLETE HISTORY OF EVERYTHING THAT THE AGENT HAS EVER PERCEIVED." So the percept sequence is all the percepts the agent has experienced up to a certain point. 

I should make sure there's no other part of the context that adds to this. The other terms like "percept" and "action" are defined, but the question is specifically about percept sequence. The learning part mentions using past percept sequences, which reinforces that it's the history of percepts. 

The answer should be concise, just the definition from the context. No need to include extra info from the learning section unless necessary. The user wants a straightforward answer based on the given context. So the answer is the complete

In [35]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Percept Sequence", rag_retriever, llm, top_k=3, min_score=0, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Percept Sequence'
Top K: 3, Score threshold: 0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.70it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer: <think>
Okay, let's see. The user is asking for the definition of "Percept Sequence" based on the provided context. Let me check the context again.

Looking at the SIMPLE TERMS section under [PAGE], there's a bullet point for PERCEPT SEQUENCE. The description says it's the "COMPLETE HISTORY OF EVERYTHING THAT THE AGENT HAS EVER PERCEIVED." So the answer should be straightforward. I need to make sure I'm not missing any other parts. The context also mentions that learning involves using past percept sequences, but the question is specifically about the definition. So the answer is just the definition given there. I should present it concisely as per the instructions.
</think>

Answer: A percept sequence is the complete history of everything that the agent has ever perceived.
Sources: [{'source': '3.L4_L10 Module 2_Intelligent Agents Final.pdf', 'page': 3, 'score': 0.050307273864746094, 'preview': 'SIMPLE TERMS  -- [PAGE]\n• PERCEPT\n• AGENT’S PERCEPTUAL INPUTS AT ANY GIVEN INSTA